# 🍷 Construcción del Modelo de Clasificación de Vinos
### Machine Learning con SVM — Dataset Wine (scikit-learn)

**Flujo de trabajo:**  
`Datos → Preparación → Pipeline → Entrenamiento → Evaluación → Joblib → modelo_wine.pkl`

---


## Paso 1 — Importar las librerías

Importamos todas las herramientas necesarias para:
manipulación de datos, visualización, división, escalamiento,
construcción del Pipeline, entrenamiento del SVM, evaluación y almacenamiento del modelo.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print("✅ Librerías importadas correctamente")


## Paso 2 — Cargar el dataset Wine

- Cargamos el dataset **Wine** disponible en scikit-learn.
- Convertimos los datos a un DataFrame de Pandas.
- Agregamos la variable objetivo (`target`).
- Mostramos los primeros registros.


In [ ]:
wine = load_wine()

df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

print("Dataset cargado ✅")
print(f"Forma del DataFrame: {df.shape}")
df.head()


## Paso 3 — Asignar nombres a las clases

Las clases originales (0, 1, 2) se mapean a etiquetas didácticas:

| Clase original | Etiqueta |
|:-:|:-:|
| 0 | Cabernet |
| 1 | Merlot |
| 2 | Pinot Noir |

> **Nota:** estas etiquetas se usan únicamente con fines didácticos.


In [ ]:
etiquetas = {0: 'Cabernet', 1: 'Merlot', 2: 'Pinot Noir'}
df['tipo_vino'] = df['target'].map(etiquetas)

print("Columna 'tipo_vino' creada ✅")
df[['target', 'tipo_vino']].head(8)


## Paso 4 — Exploración inicial de los datos

Analizamos:
- Dimensiones y tipos de datos
- Valores nulos
- Estadísticas descriptivas
- Distribución de registros por clase


In [ ]:
print(f"Dimensiones del dataset : {df.shape}")
print(f"Nº de variables (features): {len(wine.feature_names)}")


In [ ]:
print("Tipos de datos:")
print(df.dtypes)


In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())


In [ ]:
print("Estadísticas descriptivas:")
df.describe().round(2)


In [ ]:
print("Registros por tipo de vino:")
print(df['tipo_vino'].value_counts())

df['tipo_vino'].value_counts().plot(
    kind='bar', color=['#6C5CE7','#A29BFE','#DFE6E9'],
    edgecolor='white', figsize=(6, 4)
)
plt.title('Distribución de clases')
plt.xlabel('Tipo de vino')
plt.ylabel('Cantidad de registros')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 🔍 Respuestas a las preguntas de análisis

| Pregunta | Respuesta |
|---|---|
| ¿Existen valores faltantes? | **No** — todas las columnas tienen 0 nulos |
| ¿Las tres clases tienen la misma cantidad de registros? | **No** — Merlot: 71, Cabernet: 59, Pinot Noir: 48 |
| ¿Se observa diferencia entre escalas de las variables? | **Sí** — `proline` ronda los 750, mientras `nonflavanoid_phenols` ronda 0.36 |


## Paso 5 — Selección de características

Utilizamos únicamente las cuatro características indicadas:

| Variable | Descripción |
|---|---|
| `alcohol` | Contenido de alcohol |
| `malic_acid` | Ácido málico |
| `color_intensity` | Intensidad del color |
| `proline` | Concentración de prolina |


In [ ]:
features = ['alcohol', 'malic_acid', 'color_intensity', 'proline']

X = df[features]
y = df['tipo_vino']

print(f"X.shape: {X.shape}")
print(f"y.shape: {y.shape}")
X.head()


## Paso 6 — Separación de los datos

Dividimos en **80 % entrenamiento / 20 % prueba** con división estratificada.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}  |  y_test: {y_test.shape}")


### 🔍 Respuestas a las preguntas de análisis

| Pregunta | Respuesta |
|---|---|
| ¿Por qué no usar el 100 % para entrenar? | Para evaluar la **capacidad de generalización** del modelo en datos que nunca vio. Si entrenamos con todo, no podemos saber si el modelo aprende o memoriza. |
| ¿Qué función cumple la estratificación? | Garantiza que la **proporción de cada clase** sea representativa tanto en train como en test, evitando sesgos en la evaluación. |


## Paso 7 — Construcción del Pipeline

Creamos un Pipeline con dos etapas:
1. `StandardScaler` — normaliza las variables
2. `SVC` — clasificador SVM con kernel lineal

> **¿Por qué incluir el escalamiento dentro del Pipeline?**  
> Evita *data leakage*: el scaler aprende la media y desviación **solo** con los datos de entrenamiento y aplica esa misma transformación al conjunto de prueba. Si se escala antes de dividir, el modelo "ve" información del test indirectamente.


In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc',    SVC(kernel='linear', random_state=42))
])

print("Pipeline creado ✅")
print(pipeline)


## Paso 8 — Entrenamiento del modelo


In [ ]:
pipeline.fit(X_train, y_train)

print("Modelo entrenado ✅")
print(f"Clases aprendidas: {pipeline.classes_}")


## Paso 9 — Predicciones sobre el conjunto de prueba


In [ ]:
y_pred = pipeline.predict(X_test)

comparacion = pd.DataFrame({
    'Real'    : y_test.values,
    'Predicho': y_pred
}).reset_index(drop=True)

print("Comparación Real vs Predicho (primeros 10 registros):")
comparacion.head(10)


## Paso 10 — Evaluación del modelo


In [ ]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec  = recall_score(y_test, y_pred, average='weighted')
f1   = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")


In [ ]:
print("Reporte completo por clase:\n")
print(classification_report(y_test, y_pred,
                             target_names=['Cabernet', 'Merlot', 'Pinot Noir']))


## Paso 11 — Matriz de confusión


In [ ]:
class_names = ['Cabernet', 'Merlot', 'Pinot Noir']
cm = confusion_matrix(y_test, y_pred, labels=class_names)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Matriz de Confusión — Clasificación de Vinos', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

print(f"\nPredicciones correctas (diagonal): {[cm[i,i] for i in range(3)]}")
print(f"Errores totales                   : {cm.sum() - cm.trace()}")


### 🔍 Análisis de la matriz

| | Cabernet | Merlot | Pinot Noir |
|---|:-:|:-:|:-:|
| **Cabernet** | ✅ 12 | 0 | 0 |
| **Merlot** | 0 | ✅ 13 | 1 |
| **Pinot Noir** | 0 | 2 | ✅ 8 |

- **Cabernet** se clasifica perfectamente.
- **Merlot** y **Pinot Noir** presentan ligera confusión entre sí (3 errores en total).


## Paso 12 — Predicción para un nuevo vino

Probamos el modelo con una observación nueva:

| Alcohol | Malic Acid | Color Intensity | Proline |
|:-:|:-:|:-:|:-:|
| 13.5 | 1.8 | 5.2 | 1000 |


In [ ]:
nuevo_vino = pd.DataFrame({
    'alcohol'         : [13.5],
    'malic_acid'      : [1.8],
    'color_intensity' : [5.2],
    'proline'         : [1000]
})

prediccion_nuevo = pipeline.predict(nuevo_vino)
print(f"Tipo de vino predicho: {prediccion_nuevo[0]}")


## Paso 13 — Guardar el modelo con Joblib


In [ ]:
joblib.dump(pipeline, 'modelo_wine.pkl')

import os
print(f"Archivo guardado  : modelo_wine.pkl ✅")
print(f"Existe en disco   : {os.path.exists('modelo_wine.pkl')}")
print(f"Tamaño            : {os.path.getsize('modelo_wine.pkl')} bytes")


## Paso 14 — Recuperar el modelo y verificar


In [ ]:
modelo_cargado = joblib.load('modelo_wine.pkl')
prediccion_recuperada = modelo_cargado.predict(nuevo_vino)

print(f"Predicción antes de guardar : {prediccion_nuevo[0]}")
print(f"Predicción tras recuperar   : {prediccion_recuperada[0]}")
print(f"¿Resultados iguales?        : {prediccion_nuevo[0] == prediccion_recuperada[0]} ✅")


---

## 💡 Reflexión final

### Diferencia entre entrenar y utilizar un modelo

| | Entrenar un modelo | Utilizar un modelo guardado |
|---|---|---|
| **¿Qué hace?** | Ajusta los parámetros del SVM a partir de los datos de entrenamiento | Carga los parámetros ya ajustados desde disco |
| **¿Necesita datos etiquetados?** | ✅ Sí | ❌ No |
| **¿Consume tiempo de cómputo?** | ✅ Sí | Mínimo |
| **¿Cuándo se usa?** | Una sola vez (o al reentrenar) | Cada vez que se quiere predecir |

### Flujo completo de la práctica

```
Datos (Wine dataset)
  └─► Preparación (DataFrame + tipo_vino)
        └─► Pipeline (StandardScaler + SVC linear)
              └─► Entrenamiento (fit con X_train)
                    └─► Evaluación (accuracy 91.67 %)
                          └─► Joblib (dump)
                                └─► modelo_wine.pkl ✅
```
